# Supplementary Tables

Generates all supplementary tables as CSV files in `paper/final/tables/`.

1. Table S-GLMcovariates — GLM spatial spread covariates
2. Table S-PersistencePredictors — Within-state persistence predictors
3. Table S-CaseOutbreakData — USDA HPAI case data summary
4. Table S-ABCposterior — ABC-SMC accepted parameter summary
5. Table S-NetworkTraits — County-level network metrics
6. Table S-ICVIrecent — ICVI interstate cattle movement matrix
7. Table S-SimulationScenarios — Counterfactual scenario outcomes

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJ_ROOT = Path('../..').resolve()
PAPER_DIR = PROJ_ROOT / 'paper' / 'final'
DATA_DIR = PROJ_ROOT
TABLE_DIR = PAPER_DIR / 'tables'
TABLE_DIR.mkdir(exist_ok=True)
RESULTS_DIR = PAPER_DIR / 'analysis' / 'simulations' / 'results'
print(f'Output: {TABLE_DIR}')

## Table S-GLMcovariates

In [ ]:
glm_covariates = pd.DataFrame([
    ['Cattle movement', 'USAMMv3 cattle movement', 'Pairwise',
     'Number of dairy cattle shipped between each state pair per year, averaged across 1,000 stochastic network realizations of the US Animal Movement Model (Sellman et al., 2022).',
     'USAMMv3'],
    ['Cattle movement', 'ICVI cattle movement', 'Pairwise',
     'Total interstate dairy cattle shipments between each state pair from regulatory certificates of veterinary inspection.',
     'ICVI records (2015\u20132016)'],
    ['Dairy industry', 'Dairy inventory (origin)', 'Origin',
     'Total number of dairy cows in the origin state.',
     'USDA NASS (2024)'],
    ['Dairy industry', 'Dairy inventory (destination)', 'Destination',
     'Total number of dairy cows in the destination state.',
     'USDA NASS (2024)'],
    ['Dairy industry', 'Milk production per cow (origin)', 'Origin',
     'Annual milk production divided by dairy cow inventory in the origin state. Proxy for production intensity.',
     'USDA NASS (2024)'],
    ['Dairy industry', 'Milk production per cow (destination)', 'Destination',
     'Annual milk production divided by dairy cow inventory in the destination state.',
     'USDA NASS (2024)'],
    ['Dairy industry', 'Number of herds (origin)', 'Origin',
     'Number of dairy operations in the origin state.',
     'USDA NASS (2024)'],
    ['Dairy industry', 'Number of herds (destination)', 'Destination',
     'Number of dairy operations in the destination state.',
     'USDA NASS (2024)'],
    ['Dairy industry', 'Replacement ratio (origin)', 'Origin',
     'Proportion of calved dairy cattle relative to total dairy cattle in the origin state. Proxy for herd turnover rate.',
     'USDA NASS (2024)'],
    ['Dairy industry', 'Replacement ratio (destination)', 'Destination',
     'Proportion of calved dairy cattle relative to total dairy cattle in the destination state.',
     'USDA NASS (2024)'],
    ['Geography', 'Geographic distance', 'Pairwise',
     'Great-circle distance between population-weighted state centroids (km).',
     'Calculated'],
    ['Sampling', 'Sampling bias residual (origin)', 'Origin',
     'Residuals from a regression of sequenced genomes against dairy inventory, outbreak counts, and number of herds in the origin state.',
     'Calculated'],
    ['Sampling', 'Sampling bias residual (destination)', 'Destination',
     'Residuals from a regression of sequenced genomes against dairy inventory, outbreak counts, and number of herds in the destination state.',
     'Calculated'],
], columns=['Category', 'Predictor', 'Type', 'Description', 'Source'])

glm_covariates.to_csv(TABLE_DIR / 'Table_S_GLMcovariates.csv', index=False)
print(f'Table S-GLMcovariates: {len(glm_covariates)} predictors')
glm_covariates

## Table S-PersistencePredictors

In [ ]:
persistence_predictors = pd.DataFrame([
    ['Within-state network', 'Internal volume',
     'Total number of dairy cattle shipped within a state per year, summed across all within-state county-to-county routes.',
     'USAMMv3 (1,000 network realizations)'],
    ['Within-state network', 'Internal density',
     'Fraction of realized within-state shipping routes relative to the maximum possible directed edges between dairy counties in the state.',
     'USAMMv3 (1,000 network realizations)'],
    ['Within-state network', 'Dairy counties',
     'Number of counties with at least one dairy premise in the state.',
     'USDA NASS (2022)'],
    ['Interstate network', 'Incoming routes',
     'Number of unique interstate shipping routes directed into the state (distinct origin–destination county pairs crossing a state border).',
     'USAMMv3 (1,000 network realizations)'],
    ['Interstate network', 'Incoming volume',
     'Total number of dairy cattle shipped into the state from other states per year, summed across all incoming interstate routes.',
     'USAMMv3 (1,000 network realizations)'],
    ['Dairy industry', 'Herd density',
     'Number of dairy cows per square kilometer of state land area.',
     'USDA NASS (2024)'],
    ['Dairy industry', 'Replacement ratio',
     'Proportion of calved dairy cattle relative to total dairy cattle in the state; proxy for herd turnover rate.',
     'USDA NASS (2024)'],
], columns=['Category', 'Predictor', 'Description', 'Data source'])

persistence_predictors.to_csv(TABLE_DIR / 'Table_S_PersistencePredictors.csv', index=False)
print(f'Table S-PersistencePredictors: {len(persistence_predictors)} predictors')
persistence_predictors

## Table S-CaseOutbreakData

In [ ]:
outbreaks_raw = pd.read_csv(DATA_DIR / 'data' / 'raw' / 'USDA_h5n1_cattle_poultry_outbreaks_2024-01-01_to_2025-12-21.csv')
cattle = outbreaks_raw[outbreaks_raw['Species'] == 'Cattle'].copy()
cattle['Date'] = pd.to_datetime(cattle['Confirmation Date'], format='mixed', dayfirst=False)

# Remove D1.1 states (Arizona, Wisconsin)
cattle = cattle[~cattle['State'].isin(['Arizona', 'Wisconsin'])]

summary = cattle.groupby('State').agg(
    n_cases=('Special Id', 'count'),
    first_confirmation=('Date', 'min'),
).reset_index()

summary['first_confirmation'] = summary['first_confirmation'].dt.strftime('%Y-%m-%d')
summary = summary.sort_values('first_confirmation').reset_index(drop=True)
summary.columns = ['State', 'Confirmed cases', 'First confirmation']

total_row = pd.DataFrame([{
    'State': 'Total',
    'Confirmed cases': summary['Confirmed cases'].sum(),
    'First confirmation': summary['First confirmation'].min(),
}])
summary = pd.concat([summary, total_row], ignore_index=True)

summary.to_csv(TABLE_DIR / 'Table_S_CaseOutbreakData.csv', index=False)
print(f'Table S-CaseOutbreakData: {len(summary)-1} states + total')
summary

## Table S-ABCposterior

In [ ]:
abc = pd.read_csv(RESULTS_DIR / 'baseline' / 'abc_accepted.csv')

params = {
    'beta': ('\u03b2', 'Per-head transmission coefficient', '[0.001, 0.05]'),
    'alpha': ('\u03b1', 'Density scaling exponent', '[0, 1]'),
    'gamma': ('\u03b3', 'Premises scaling exponent', '[0, 1]'),
    'mu_c': ('\u03bc_c', 'Base recovery period (days)', '[5, 60]'),
    'npi_factor': ('f_NPI', 'NPI reduction factor', '[0, 1]'),
    'tmrca_offset': ('t_offset', 'tMRCA offset (days before 2024-03-25)', '[60, 150]'),
}

rows = []
for col, (symbol, desc, prior) in params.items():
    vals = abc[col]
    rows.append({
        'Parameter': symbol,
        'Description': desc,
        'Prior': f'Uniform{prior}',
        'Median': f'{vals.median():.4f}',
        '95% CI': f'[{vals.quantile(0.025):.4f}, {vals.quantile(0.975):.4f}]',
        'Min': f'{vals.min():.4f}',
        'Max': f'{vals.max():.4f}',
    })

abc_table = pd.DataFrame(rows)
abc_table.to_csv(TABLE_DIR / 'Table_S_ABCposterior.csv', index=False)
print(f'Table S-ABCposterior: {len(abc_table)} parameters from {len(abc)} accepted sets')
abc_table

## Table S-NetworkTraits

In [ ]:
network_traits = pd.DataFrame([
    ['Out-degree',
     'Number of unique counties to which a county ships cattle.'],
    ['In-degree',
     'Number of unique counties from which a county receives cattle shipments.'],
    ['PageRank',
     'Iterative measure of node importance on the directed, volume-weighted network; a county ranks high if it receives shipments from other high-ranking counties (Brin & Page, 1998).'],
    ['Temporal PageRank',
     'PageRank computed independently on each of 52 weekly network snapshots and averaged, capturing importance across the full temporal dynamics of cattle movement.'],
    ['Eigenvector centrality',
     'Leading eigenvector of the adjacency matrix of the undirected, volume-weighted network; scores reflect recursive importance within the densely connected network core.'],
    ['Betweenness centrality',
     'Fraction of all shortest paths (volume-weighted) between county pairs that pass through a given county; high values indicate bridge or bottleneck nodes.'],
    ['Temporal betweenness',
     'Fraction of time-respecting shortest paths passing through a county, where edges must follow temporal ordering; captures a county’s role as a temporal bridge.'],
    ['Closeness centrality',
     'Inverse of the mean shortest-path distance from a county to all other counties in the directed network; high values indicate counties that can reach the rest of the network in fewer steps.'],
    ['Shipping consistency',
     'Fraction of days per year (0–1) on which a county has at least one outbound shipment; values near 1 indicate year-round activity.'],
    ['Max shipping gap',
     'Longest interval (days) between consecutive outbound shipments; captures the worst-case interruption in a county’s potential to export infection.'],
    ['Temporal reachability (3-hop, 30-day)',
     'Number of unique counties reachable within 30 days via 3 or fewer time-respecting shipment hops; directly operationalizes short-term epidemic potential.'],
    ['Clustering coefficient',
     'Fraction of possible triangles present among a county’s neighbors (undirected); high values indicate tightly knit local trading subnetworks.'],
], columns=['Metric', 'Description'])

network_traits.to_csv(TABLE_DIR / 'Table_S_NetworkTraits.csv', index=False)
print(f'Table S-NetworkTraits: {len(network_traits)} metrics')
network_traits

## Table S-ICVIrecent

In [ ]:
# Full ICVI dairy cattle movement data (all 48 states)
icvi_full = pd.read_csv(DATA_DIR / 'data' / 'raw' / 'st_full_dairy_matrix.csv', index_col=0)
icvi_partial = pd.read_csv(DATA_DIR / 'data' / 'raw' / 'st_partial_dairy_matrix.csv', index_col=0)

print(f'Full ICVI matrix: {icvi_full.shape[0]}x{icvi_full.shape[1]} states')
print(f'Partial ICVI matrix: {icvi_partial.shape[0]}x{icvi_partial.shape[1]} states')
print(f'Total shipments (full): {int(icvi_full.values.sum()):,}')
print(f'Total shipments (partial): {int(icvi_partial.values.sum()):,}')

# Summary by state
icvi_summary = pd.DataFrame({
    'State': icvi_full.index,
    'Outgoing shipments (full)': icvi_full.sum(axis=1).astype(int).values,
    'Incoming shipments (full)': icvi_full.sum(axis=0).astype(int).values,
    'Outgoing shipments (partial)': icvi_partial.sum(axis=1).astype(int).values,
    'Incoming shipments (partial)': icvi_partial.sum(axis=0).astype(int).values,
})
icvi_summary = icvi_summary.sort_values('Outgoing shipments (full)', ascending=False).reset_index(drop=True)
icvi_summary.to_csv(TABLE_DIR / 'Table_S_ICVIrecent.csv', index=False)
print(f'\nTable S-ICVIrecent: {len(icvi_summary)} states')
icvi_summary.head(10)

## Table S-SimulationScenarios

In [ ]:
# Load all baseline scenario results
baseline_files = {
    'No NPI': 'no_npi_results.csv',
    'Federal order (day 35)': 'npi_day35_results.csv',
    'Day 0, 90% reduction': 'day0_f010_results.csv',
    'Day 16, 90% reduction': 'day16_f010_results.csv',
    'tMRCA + 30 days': 'tmrca_30d_results.csv',
    'tMRCA + 60 days': 'tmrca_60d_results.csv',
    '10% interstate reduction': 'strength_f090_results.csv',
    '30% interstate reduction': 'strength_f070_results.csv',
    '50% interstate reduction': 'strength_f050_results.csv',
    '70% interstate reduction': 'strength_f030_results.csv',
    '90% interstate reduction': 'strength_f010_results.csv',
    '90% interstate + 90% intrastate': 'within90_between90_results.csv',
}

rows = []
for label, fname in baseline_files.items():
    fpath = RESULTS_DIR / 'baseline' / fname
    if not fpath.exists():
        print(f'  MISSING: {fname}')
        continue
    df = pd.read_csv(fpath)
    rows.append({
        'Category': 'NPI counterfactual',
        'Scenario': label,
        'N simulations': f'{len(df):,}',
        'Median states': f'{df["n_states"].median():.0f}',
        'States [2.5%, 97.5%]': f'[{df["n_states"].quantile(0.025):.0f}, {df["n_states"].quantile(0.975):.0f}]',
        'Median counties': f'{df["n_counties"].median():.0f}',
        'Counties [2.5%, 97.5%]': f'[{df["n_counties"].quantile(0.025):.0f}, {df["n_counties"].quantile(0.975):.0f}]',
    })

# Containment scenarios — only perimeters used in the paper
containment_dir = RESULTS_DIR / 'containment_v3'
perimeter_labels = {
    'panhandle': 'Panhandle (14 counties)',
    'geo_ring2': 'Geographic ring 2 (47 counties)',
    'geo_ring4': 'Geographic ring 4 (111 counties)',
    'net99_1hop': '1-link network perimeter 99% (41 counties)',
    'net99_2hop_cond': '2-link network perimeter 99% conditional (102 counties)',
    'tx_region7': 'TX + 6 border states (437 counties)',
}
timing_labels = {
    'tmrca14': 'tMRCA + 14 days',
    'tmrca30': 'tMRCA + 30 days',
    'tmrca60': 'tMRCA + 60 days',
    'detect': 'Detection date',
}

n_missing = 0
for perimeter, p_label in perimeter_labels.items():
    for timing, t_label in timing_labels.items():
        fname = f'{perimeter}_halt_{timing}_results.csv'
        fpath = containment_dir / fname
        if not fpath.exists():
            print(f'  PENDING (darwin): {fname}')
            n_missing += 1
            continue
        df = pd.read_csv(fpath)
        rows.append({
            'Category': 'Targeted containment',
            'Scenario': f'{p_label}, {t_label}',
            'N simulations': f'{len(df):,}',
            'Median states': f'{df["n_states"].median():.0f}',
            'States [2.5%, 97.5%]': f'[{df["n_states"].quantile(0.025):.0f}, {df["n_states"].quantile(0.975):.0f}]',
            'Median counties': f'{df["n_counties"].median():.0f}',
            'Counties [2.5%, 97.5%]': f'[{df["n_counties"].quantile(0.025):.0f}, {df["n_counties"].quantile(0.975):.0f}]',
        })

scenario_table = pd.DataFrame(rows)
scenario_table.to_csv(TABLE_DIR / 'Table_S_SimulationScenarios.csv', index=False)
print(f'\nTable S-SimulationScenarios: {len(scenario_table)} scenarios ({n_missing} pending from darwin)')
scenario_table

## Summary

In [ ]:
import os
print('Tables generated:')
for f in sorted(TABLE_DIR.glob('Table_S_*.csv')):
    size = os.path.getsize(f)
    df = pd.read_csv(f)
    print(f'  {f.name}: {len(df)} rows, {size:,} bytes')